# Stance Detection — Colab Runner

**How to use:**
1. Run **Cell 1** once per session — clones the repo and installs dependencies.
2. Run **Cell 2** once per session — loads all models into memory.
3. Run **Cell 3** as many times as you like — edit `INPUT_TEXT` and `INPUT_TARGETS` and re-run without reloading models.

Use a GPU runtime (`Runtime → Change runtime type → T4 GPU`) for faster inference.

In [ ]:
# Clone repo & install deps
# Run once per session.

import subprocess, sys, os

REPO_URL = "https://github.com/jayantt23/poli-stance.git"
REPO_DIR = "poli-stance"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Add src/ to Python path
SRC_PATH = os.path.join(os.getcwd(), REPO_DIR, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Install Python dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "torch", "spacy", "huggingface_hub"],
    check=True
)

# Download spaCy English model
subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--quiet"],
    check=True
)

print("Setup complete.")

In [ ]:
#Load all models 
# Run once per session. Models are cached — re-running this cell is a no-op.

import torch
from stance.model_cache import get_nlp, get_zero_shot_pipeline

# Use GPU if available
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU (cuda:0)' if DEVICE == 0 else 'CPU'}")

print("Loading spaCy NLP model...")
nlp = get_nlp()

print("Loading zero-shot classification model (facebook/bart-large-mnli)...")
clf = get_zero_shot_pipeline(device=DEVICE)

print("All models loaded.")

In [ ]:
#  Run stance detection
# Edit INPUT_TEXT and INPUT_TARGETS, then re-run as many times as needed.

import json
from stance.stance_service import run_stance_pipeline


INPUT_TEXT = """
Prime Minister Modi's economic reforms have received mixed reactions.
While BJP supporters praise the demonetisation drive as a bold anti-corruption move,
opposition leaders like Rahul Gandhi have criticised it as an economic disaster
that hurt ordinary citizens. The Congress party has demanded a full rollback,
while AAP's Kejriwal called it a failure of governance.
"""

# Provide specific targets, or leave as [] to rely purely on auto-detection
INPUT_TARGETS = ["Modi", "Rahul Gandhi", "BJP", "Congress"]

# Optional: "india", "us", or None for no filter
COUNTRY_FILTER = "india"

# ── RUN ────────────────────────────────────────────────────────────────────────
result = run_stance_pipeline(
    text=INPUT_TEXT,
    clf=clf,
    nlp=nlp,
    targets=INPUT_TARGETS,
    # registry defaults to TARGET_REGISTRY (the full production registry)
    do_ner_target_suggestion=True,
    retrieval_mode="strict",
    top_k_evidence=3,
    include_neutral=True,
    mixed_margin=0.08,
    country_filter=COUNTRY_FILTER,
    text_id="colab_run_01",
)


print(f"Requested targets : {result['requested_targets']}")
print(f"Resolved targets  : {result['resolved_requested_targets']}")
print(f"Auto-detected     : {result['extra_entities']}")
print()

print("=" * 60)
print("STANCE RESULTS")
print("=" * 60)
for r in result["all_results"]:
    print(f"\nTarget : {r['target']}")
    print(f"Label  : {r['label']}")
    if r["scores"]:
        scores_str = ", ".join(f"{k}={v:.3f}" for k, v in r["scores"].items())
        print(f"Scores : {scores_str}")
    print(f"Used   : {r['num_sentences_used']} sentence(s)")
    if r["evidence"]:
        print("Evidence:")
        for i, ev in enumerate(r["evidence"], 1):
            print(f"  [{i}] {ev}")

print()
print("Full JSON output:")
print(json.dumps(result, indent=2))